# 🪙 Crypto Trading Bot – Walk-Forward ML Backtest (BTC + ETH)
**Data:** Yahoo Finance daily candles – BTC (Sep 2014 se) aur ETH (Nov 2017 se)

**Bot ka idea:** roz subah model bolta hai *"kal price upar jayegi ya neeche?"*. Upar jaye toh coin hold karo, warna cash mein raho.

**Is project mein:**
1. **EDA** – price, returns, volatility, drawdown, BTC-ETH correlation
2. **Features** – moving averages, RSI, volatility, momentum (sirf past data se, future nahi)
3. **ML** – Logistic Regression, Decision Tree, Random Forest (80/20 time-based split) + **Time-series cross-validation**
4. **Walk-forward bot** – model har 30 din mein naye data par dobara seekhta hai ("bot seekhta rahe")
5. **Backtest** – fees ke saath, aur **Buy & Hold**, **Moving Average** aur **DCA** se comparison
6. **Mini bucket** – BTC + ETH ke beech paisa baantna (baad mein top-5 coins tak badha sakte hain)

⚠️ Ye seekhne ka project hai, financial advice nahi. Backtest asli trading ka guarantee nahi hota.
Code easy hai: **pandas + numpy** zyada, metrics **sklearn.metrics** se (aur ek jagah formula haath se bhi check kiya), seaborn charts, chhote chhote cells, **koi lambda nahi**, loop sirf jahan zaroori ho.

In [1]:
import numpy as np                                                # numbers aur maths ke liye
import pandas as pd                                               # data table (DataFrame) ke liye
import matplotlib.pyplot as plt                                   # charts ka base library
import seaborn as sns                                             # sundar charts ke liye
import joblib, os                                                 # model save karne aur folder banane ke liye

from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit  # split aur time-series cross-validation ke liye
from sklearn.linear_model import LogisticRegression               # Logistic Regression model
from sklearn.tree import DecisionTreeClassifier                   # Decision Tree model
from sklearn.ensemble import RandomForestClassifier               # Random Forest model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score  # score nikalne ke liye
from sklearn.metrics import confusion_matrix, classification_report  # confusion matrix aur poori report ke liye

sns.set_theme(style="whitegrid", palette="deep")                  # seaborn ka look set karo
pd.set_option("display.width", 200)                               # table ki width badhao
os.makedirs("models", exist_ok=True)                              # models ke liye folder banao

# 💡 INSIGHT: Sirf standard libraries use hui: pandas, numpy, seaborn aur sklearn (models + metrics). Koi special trading library
#    nahi, taaki code easy aur samajhne layak rahe.

In [2]:
import yfinance as yf                                              # Yahoo Finance se data laane ka library

btc = yf.download("BTC-USD", start="2014-01-01", auto_adjust=False)  # BTC ka daily data
btc.columns = btc.columns.get_level_values(0)                      # column ke naam saaf karo
btc.columns.name = None                                            # extra naam hatao
btc.to_csv("BTC-USD.csv")                                          # CSV file save karo

eth = yf.download("ETH-USD", start="2014-01-01", auto_adjust=False)  # ETH ka daily data
eth.columns = eth.columns.get_level_values(0)                      # column ke naam saaf karo
eth.columns.name = None                                            # extra naam hatao
eth.to_csv("ETH-USD.csv")                                          # CSV file save karo

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


## 2. Data load aur quick checks

In [3]:
btc = pd.read_csv("BTC-USD.csv", index_col="Date", parse_dates=True)   # BTC ki CSV padho, Date ko index banao
eth = pd.read_csv("ETH-USD.csv", index_col="Date", parse_dates=True)   # ETH ki CSV padho, Date ko index banao

In [4]:
btc.columns

Index(['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='str')

In [5]:
eth.columns

Index(['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='str')

In [6]:
btc = btc[["Open", "High", "Low", "Close", "Volume"]].sort_index()    # BTC ke zaroori columns rakho, date ke order mein
eth = eth[["Open", "High", "Low", "Close", "Volume"]].sort_index()    # ETH ke zaroori columns rakho, date ke order mein

In [7]:
check = pd.DataFrame({                                                 # dono coins ka checkup table
    "coin": ["BTC", "ETH"],                                            # coin ke naam
    "rows": [len(btc), len(eth)],                                      # kitni rows hain
    "start": [btc.index.min().date(), eth.index.min().date()],         # data kab se shuru
    "end": [btc.index.max().date(), eth.index.max().date()],           # data kab tak
    "missing": [btc.isna().sum().sum(), eth.isna().sum().sum()],       # kitni missing values
    "dup_dates": [btc.index.duplicated().sum(), eth.index.duplicated().sum()],  # kitni duplicate dates
    "gap_days": [(btc.index.to_series().diff().dt.days > 1).sum(), (eth.index.to_series().diff().dt.days > 1).sum()],  # kitne din ka gap
    "ohlc_ok_%": [((btc["Low"] <= btc["Close"]) & (btc["Close"] <= btc["High"])).mean() * 100,  # BTC: Low <= Close <= High sahi hai?
                  ((eth["Low"] <= eth["Close"]) & (eth["Close"] <= eth["High"])).mean() * 100]  # ETH: Low <= Close <= High sahi hai?
}).round(1)            

check                                                                  # checkup table dikhao

# 💡 INSIGHT: BTC ke 4,441 din (17 Sep 2014 se) aur ETH ke 3,292 din (9 Nov 2017 se). Dono mein koi missing value,
#    duplicate ya gap nahi, aur 100% rows mein Low <= Close <= High sahi hai. Data saaf hai.

## 3. EDA – data ko samjho
Pehle dono coins ki price, daily returns aur volatility dekhte hain.zx

In [10]:
prices = pd.concat([btc['Close'].rename('BTC'), eth['Close'].rename('ETH')],axis=1)
rets = prices.pct_change()
vol = rets.rolling(30).std() * np.sqrt(365) * 100 

In [ ]:
prices = pd.concat([btc["Close"].rename("BTC"), eth["Close"].rename("ETH")], axis=1)  # dono ki closing price ek table mein
rets = prices.pct_change()                                        # daily return (%)
vol = rets.rolling(30).std() * np.sqrt(365) * 100                 # 30 din ki saalana volatility (%)